In [ ]:
# 1. IMPORT LIBRARY LENGKAP
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 2. LOAD DATASET

df = pd.read_csv('../dataset/predictive_maintenance.csv')

print("=== 5 Baris Pertama Dataset ===")
display(df.head())
print("=== Info Dataset ===")
df.info()

In [ ]:
# 3. PREPROCESSING
print("=== Memulai Preprocessing ===")

# Buang kolom yang tidak relevan
df_bersih = df.drop(columns=['UDI', 'Product ID', 'Target'])

# Ubah teks 'Type' jadi angka
le_type = LabelEncoder()
df_bersih['Type'] = le_type.fit_transform(df_bersih['Type'])

# Ubah teks 'Failure Type' (Target) jadi angka 0-5 (Wajib untuk XGBoost)
le_target = LabelEncoder()
df_bersih['Failure Type'] = le_target.fit_transform(df_bersih['Failure Type'])

# Pisahkan Input/Label (X) dan Output/Target (y)
X = df_bersih.drop(columns=['Failure Type'])
y = df_bersih['Failure Type']

print("Preprocessing Selesai! Data siap dibagi.")
display(df_bersih.head())

In [ ]:
# 4. EDA: GRAFIK DISTRIBUSI KELAS KERUSAKAN
plt.figure(figsize=(10, 5))

# Kita pakai teks asli untuk grafik agar mudah dibaca
kelas_teks = le_target.inverse_transform(df_bersih['Failure Type'])

sns.countplot(y=kelas_teks, 
              order=pd.Series(kelas_teks).value_counts().index, 
              palette='viridis')

plt.title('Distribusi 6 Kelas Kondisi Mesin', fontweight='bold')
plt.xlabel('Jumlah Data')
plt.ylabel('Kondisi Asli')
plt.show()

In [ ]:
# 5A. PEMBAGIAN DATA RASIO 70:30
print("=== MENYIAPKAN DATA 70:30 ===")
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(X, y, test_size=0.3, random_state=42)

scaler_70 = StandardScaler()
X_train_70_scaled = scaler_70.fit_transform(X_train_70)
X_test_70_scaled = scaler_70.transform(X_test_70)

print(f"Data 70:30 Berhasil Dibuat!")
print(f"Jumlah Data Training: {X_train_70.shape[0]} baris")
print(f"Jumlah Data Testing  : {X_test_70.shape[0]} baris\n")

In [ ]:
# 5B. PEMBAGIAN DATA RASIO 80:20
print("=== MENYIAPKAN DATA 80:20 ===")
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_80 = StandardScaler()
X_train_80_scaled = scaler_80.fit_transform(X_train_80)
X_test_80_scaled = scaler_80.transform(X_test_80)

print(f"Data 80:20 Berhasil Dibuat!")
print(f"Jumlah Data Training: {X_train_80.shape[0]} baris")
print(f"Jumlah Data Testing  : {X_test_80.shape[0]} baris\n")

In [ ]:
# 5C. PEMBAGIAN DATA RASIO 90:10
print("=== MENYIAPKAN DATA 90:10 ===")
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(X, y, test_size=0.1, random_state=42)

scaler_90 = StandardScaler()
X_train_90_scaled = scaler_90.fit_transform(X_train_90)
X_test_90_scaled = scaler_90.transform(X_test_90)

print(f"Data 90:10 Berhasil Dibuat!")
print(f"Jumlah Data Training: {X_train_90.shape[0]} baris")
print(f"Jumlah Data Testing  : {X_test_90.shape[0]} baris\n")

In [ ]:
# 6. MELATIH 3 MODEL PADA SEMUA RASIO
models = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss', n_estimators=300, max_depth=5, learning_rate=0.1),
    "HistGradient": HistGradientBoostingClassifier(random_state=42, class_weight='balanced', max_iter=250, learning_rate=0.1, max_depth=6, max_bins=255),
    "ExtraTrees": ExtraTreesClassifier(random_state=42, class_weight='balanced', n_estimators=300, max_depth=None, max_features='sqrt', min_samples_split=4, n_jobs=-1)
}

# Mengumpulkan semua variabel split dari atas agar mudah dieksekusi bersamaan
kumpulan_data = {
    '70:30': (X_train_70_scaled, X_test_70_scaled, y_train_70, y_test_70),
    '80:20': (X_train_80_scaled, X_test_80_scaled, y_train_80, y_test_80),
    '90:10': (X_train_90_scaled, X_test_90_scaled, y_train_90, y_test_90)
}

hasil_eksperimen = []
print("=== MEMULAI PELATIHAN AI ===")

for rasio, (X_tr, X_ts, y_tr, y_ts) in kumpulan_data.items():
    print(f"Sedang menguji AI di rasio {rasio}...")
    for nama_model, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_ts)
        
        akurasi = accuracy_score(y_ts, y_pred)
        presisi = precision_score(y_ts, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_ts, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_ts, y_pred, average='weighted', zero_division=0)
        
        hasil_eksperimen.append({
            'Rasio Data': rasio, 'Algoritma': nama_model,
            'Accuracy': akurasi * 100, 'Precision': presisi * 100,
            'Recall': recall * 100, 'F1-Score': f1 * 100
        })

print("\nPelatihan Selesai! Cek hasilnya di cell bawah.")

In [ ]:
# 7. MENAMPILKAN TABEL DAN GRAFIK HASIL UJIAN AI
df_hasil = pd.DataFrame(hasil_eksperimen)

print("=== TABEL EVALUASI LENGKAP ===")
display(df_hasil.round(2).sort_values(by=['Rasio Data', 'Accuracy'], ascending=[False, False]).reset_index(drop=True))

plt.figure(figsize=(10, 6))
sns.barplot(data=df_hasil, x='Rasio Data', y='Accuracy', hue='Algoritma')

plt.title('Perbandingan Akurasi AI di Berbagai Rasio Data Ujian', fontsize=14, fontweight='bold')
plt.ylim(0, 110)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 8. SIMULASI PENGUJIAN DATA BARU DENGAN 100% DATA
print("=== SIMULASI DETEKSI KERUSAKAN MESIN ===")

# Standarisasi 100% data
scaler_final = StandardScaler()
X_scaled_full = scaler_final.fit_transform(X)

# Melatih ulang Trio AI dengan 100% data agar wawasannya penuh
models_final = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss'),
    "HistGradient": HistGradientBoostingClassifier(random_state=42, class_weight='balanced', max_iter=200),
    "ExtraTrees": ExtraTreesClassifier(random_state=42, class_weight='balanced', n_estimators=100)
}

print("🛠️ Mempersiapkan AI dengan 100% Data Pabrik...")
for nama, model in models_final.items():
    model.fit(X_scaled_full, y)

# Input Angka Baru dari Sensor
data_baru = pd.DataFrame({
    'Type': [0],                        # Kualitas Low (Rentan)
    'Air temperature [K]': [244],     
    'Process temperature [K]': [238], 
    'Rotational speed [rpm]': [123],   # Putaran mesin agak tertahan
    'Torque [Nm]': [2000],              # Tarikan/Beban SANGAT TINGGI
    'Tool wear [min]': [200]            # Alat sudah lumayan lama dipakai
})

print("\n📡 Membaca Data Sensor Masuk:")
display(data_baru)

# Standarisasi input baru dan prediksi
data_baru_scaled = scaler_final.transform(data_baru)

print("\n🚨 HASIL MASING-MASING MODEL")
for nama, model in models_final.items():
    tebakan_angka = model.predict(data_baru_scaled)
    tebakan_teks = le_target.inverse_transform(tebakan_angka)
    print(f"[{nama}] menebak: >> {tebakan_teks[0].upper()} <<")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("=== 1. PERSIAPAN AI & DATA ===")
# 1. Definisi Model
models = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss', n_estimators=300, max_depth=5, learning_rate=0.1),
    "HistGradient": HistGradientBoostingClassifier(random_state=42, class_weight='balanced', max_iter=250, learning_rate=0.1, max_depth=6, max_bins=255),
    "ExtraTrees": ExtraTreesClassifier(random_state=42, class_weight='balanced', n_estimators=300, max_depth=None, max_features='sqrt', min_samples_split=4, n_jobs=-1)
}

# 2. Murni pakai variabel aslimu yang sudah aman dari error
kumpulan_data = {
    '70:30': (X_train_70_scaled, X_test_70_scaled, y_train_70, y_test_70),
    '80:20': (X_train_80_scaled, X_test_80_scaled, y_train_80, y_test_80),
    '90:10': (X_train_90_scaled, X_test_90_scaled, y_train_90, y_test_90)
}

hasil_eksperimen = []
print("=== 2. MEMULAI PELATIHAN AI ===")

for rasio, (X_tr, X_ts, y_tr, y_ts) in kumpulan_data.items():
    print(f"Sedang menguji AI di rasio {rasio}...")
    
    # Hitung bobot manual khusus untuk XGBoost
    bobot_sampel_xgb = compute_sample_weight(class_weight='balanced', y=y_tr)
    
    for nama_model, model in models.items():
        if nama_model == "XGBoost":
            model.fit(X_tr, y_tr, sample_weight=bobot_sampel_xgb)
        else:
            model.fit(X_tr, y_tr)
            
        y_pred = model.predict(X_ts)
        
        # PERHITUNGAN METRIK (Pakai MACRO)
        akurasi = accuracy_score(y_ts, y_pred)
        presisi = precision_score(y_ts, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_ts, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_ts, y_pred, average='macro', zero_division=0)
        
        hasil_eksperimen.append({
            'Rasio Data': rasio, 'Algoritma': nama_model,
            'Accuracy': akurasi * 100, 'Precision (Macro)': presisi * 100,
            'Recall (Macro)': recall * 100, 'F1-Score (Macro)': f1 * 100
        })

print("\n=== 3. HASIL EVALUASI AKHIR ===")
df_hasil = pd.DataFrame(hasil_eksperimen)

# Menampilkan Tabel
print("--- TABEL METRIK LENGKAP ---")
display(df_hasil.round(2).sort_values(by=['Rasio Data', 'Recall (Macro)'], ascending=[False, False]).reset_index(drop=True))

# Menampilkan Grafik
plt.figure(figsize=(10, 6))
sns.barplot(data=df_hasil, x='Rasio Data', y='Recall (Macro)', hue='Algoritma', palette='viridis')

plt.title('Kemampuan AI Mendeteksi Kerusakan Mesin (Recall Macro)', fontsize=14, fontweight='bold')
plt.ylim(0, 100)
plt.ylabel('Tingkat Keberhasilan Deteksi (%)')
plt.legend(loc='lower right', title='Model AI')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Definisi label kategori (Pastikan urutannya sesuai dengan label 0-5 di dataset Anda)
label_names = ['No Failure', 'HDF', 'PWF', 'OSF', 'TWF', 'RNF']

print("=== MEMULAI GENERASI CONFUSION MATRIX UNTUK SEMUA SPLIT ===")

# Loop melalui setiap rasio data
for rasio, (X_tr, X_ts, y_tr, y_ts) in kumpulan_data.items():
    # Membuat satu baris berisi 3 kolom (untuk 3 model) per rasio
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    fig.suptitle(f"EVALUASI CONFUSION MATRIX - RASIO DATA {rasio}", fontsize=16, fontweight='bold', y=1.05)
    
    for i, (nama_model, model) in enumerate(models.items()):
        # Model melakukan prediksi pada data test sesuai rasionya
        y_pred = model.predict(X_ts)
        
        # Hitung Matrix
        cm = confusion_matrix(y_ts, y_pred)
        
        # Visualisasi dengan Heatmap
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], 
                    xticklabels=label_names, yticklabels=label_names, cbar=False)
        
        axes[i].set_title(f"Model: {nama_model}", fontsize=12)
        axes[i].set_xlabel('Prediksi AI')
        axes[i].set_ylabel('Kenyataan')

    plt.tight_layout()
    plt.show()
    print("-" * 100)